In [1]:
import pandas as pd
import sys
sys.path.append('..')

from data.emission_factors import SCOPE2_ELECTRICITY, CHEMICALS
from data.sites import CONTAMINATED_SITE

print("Bioremediation vs Fenton treatment carbon footprint comparison")
print(f"Site: {CONTAMINATED_SITE['name']}")
print(f"Volume: {CONTAMINATED_SITE['volume_m3']} m3")
print(f"Data source: {CONTAMINATED_SITE['note']}")

Bioremediation vs Fenton treatment carbon footprint comparison
Site: Coastal marine contaminated site (illustrative)
Volume: 2000 m3
Data source: Gulumbe, Cravo-Laureau & Duran (2025), IPREM CNRS UMR 5254


In [3]:
# ── TREATMENT INVENTORIES ─────────────────────────────────────────────────────
# Functional unit: 1 m3 of contaminated seawater
# Initial concentration: 50 mg/L hexadecane + 50 mg/L phenanthrene
# Source: Gulumbe, Cravo-Laureau & Duran (2025), ET&I; 40:104361

BIOREMEDIATION = {
    # INPUTS per m3 treated
    'electricity_kWh':        2.4,   # aeration at 150 rpm for 12 days
    'inoculum_mL':           60.0,   # 10% bacterial inoculum volume
    'nutrient_medium_L':      0.054, # seawater mineral medium components
    # OUTPUTS
    'hexadecane_removal_pct': 97.24, # % removed at day 12 (Gulumbe et al.)
    'phenanthrene_removal_pct': 63.44,
    # METADATA
    'duration_days':          12,
    'reference': 'Gulumbe et al. (2025), IPREM CNRS UMR 5254',
}

FENTON_TREATMENT = {
    # INPUTS per m3 treated
    'electricity_kWh':        0.5,   # mixing only
    'H2O2_kg':                0.8,   # hydrogen peroxide
    'FeSO4_kg':               0.12,  # ferrous sulfate catalyst
    'H2SO4_kg':               0.05,  # pH adjustment to 3-4
    # OUTPUTS
    'hexadecane_removal_pct': 90.0,  # literature estimate
    'phenanthrene_removal_pct': 80.0,# aromatic compounds respond well to Fenton
    'iron_sludge_kg':         0.15,  # waste requiring disposal
    # METADATA
    'duration_days':          2,     # faster but chemical-intensive
    'reference': 'Literature values for Fenton PAH treatment',
}

print("Bioremediation inventory:")
for k, v in BIOREMEDIATION.items():
    print(f"  {k:<30} {v}")
print()
print("Fenton treatment inventory:")
for k, v in FENTON_TREATMENT.items():
    print(f"  {k:<30} {v}")

Bioremediation inventory:
  electricity_kWh                2.4
  inoculum_mL                    60.0
  nutrient_medium_L              0.054
  hexadecane_removal_pct         97.24
  phenanthrene_removal_pct       63.44
  duration_days                  12
  reference                      Gulumbe et al. (2025), IPREM CNRS UMR 5254

Fenton treatment inventory:
  electricity_kWh                0.5
  H2O2_kg                        0.8
  FeSO4_kg                       0.12
  H2SO4_kg                       0.05
  hexadecane_removal_pct         90.0
  phenanthrene_removal_pct       80.0
  iron_sludge_kg                 0.15
  duration_days                  2
  reference                      Literature values for Fenton PAH treatment


In [5]:
def treatment_carbon_footprint(inventory, grid='FR_2023'):
    """
    Calculate the carbon footprint of a remediation treatment
    per m3 of contaminated water treated.
    Returns a breakdown dictionary in kgCO2eq per m3.
    """
    ef_elec = SCOPE2_ELECTRICITY[grid]   # kgCO2eq / kWh
    breakdown = {}

    # Electricity contribution — both treatments use electricity
    breakdown['electricity'] = inventory.get('electricity_kWh', 0) * ef_elec

    # Chemical inputs — only Fenton has these
    # We check if each chemical key exists in the inventory
    # before calculating — bioremediation has none so this loop
    # produces nothing for that treatment
    chemical_map = {
        'H2O2_kg':   'hydrogen_peroxide',
        'FeSO4_kg':  'ferrous_sulfate',
        'H2SO4_kg':  'sulfuric_acid',
    }
    for inventory_key, chemicals_key in chemical_map.items():
        if inventory_key in inventory:
            kg = inventory[inventory_key]
            ef = CHEMICALS[chemicals_key]
            breakdown[inventory_key] = kg * ef

    total = sum(breakdown.values())
    breakdown['TOTAL_kgCO2_per_m3'] = round(total, 4)
    return breakdown

# Calculate for both treatments
bio_fp    = treatment_carbon_footprint(BIOREMEDIATION)
fenton_fp = treatment_carbon_footprint(FENTON_TREATMENT)

print('Bioremediation carbon footprint (kgCO2eq per m3):')
for k, v in bio_fp.items():
    print(f'  {k:<30} {v:.4f}')

print()
print('Fenton treatment carbon footprint (kgCO2eq per m3):')
for k, v in fenton_fp.items():
    print(f'  {k:<30} {v:.4f}')

Bioremediation carbon footprint (kgCO2eq per m3):
  electricity                    0.1320
  TOTAL_kgCO2_per_m3             0.1320

Fenton treatment carbon footprint (kgCO2eq per m3):
  electricity                    0.0275
  H2O2_kg                        0.5200
  FeSO4_kg                       0.0492
  H2SO4_kg                       0.0090
  TOTAL_kgCO2_per_m3             0.6057


In [7]:
def compare_treatments_at_site(site_config, bio_fp, fenton_fp):
    """
    Scale treatment footprints to the full contaminated site volume.
    Compares carbon cost AND contaminant removal performance.
    """
    volume = site_config['volume_m3']

    # Scale from per m3 to full site
    bio_total_tCO2    = bio_fp['TOTAL_kgCO2_per_m3'] * volume / 1000
    fenton_total_tCO2 = fenton_fp['TOTAL_kgCO2_per_m3'] * volume / 1000

    # Carbon saving from choosing bioremediation over Fenton
    carbon_saving = fenton_total_tCO2 - bio_total_tCO2

    return {
        'site_volume_m3':        volume,
        'bio_total_tCO2':        round(bio_total_tCO2, 3),
        'fenton_total_tCO2':     round(fenton_total_tCO2, 3),
        'carbon_saving_tCO2':    round(carbon_saving, 3),
        'bio_hex_removal_pct':   site_config['bio_hexadecane_removal'],
        'fenton_hex_removal_pct': 90.0,
        'bio_phe_removal_pct':   site_config['bio_phenanthrene_removal'],
        'fenton_phe_removal_pct': 80.0,
    }

comparison = compare_treatments_at_site(CONTAMINATED_SITE, bio_fp, fenton_fp)

print(f"Site volume: {comparison['site_volume_m3']:,} m3")
print()
print(f"{'':35} {'Bioremediation':>15} {'Fenton':>10}")
print('-' * 62)
print(f"{'Carbon footprint (tCO2eq)':35} "
      f"{comparison['bio_total_tCO2']:>15.3f} "
      f"{comparison['fenton_total_tCO2']:>10.3f}")
print(f"{'Hexadecane removal (%)':35} "
      f"{comparison['bio_hex_removal_pct']:>15.2f} "
      f"{comparison['fenton_hex_removal_pct']:>10.1f}")
print(f"{'Phenanthrene removal (%)':35} "
      f"{comparison['bio_phe_removal_pct']:>15.2f} "
      f"{comparison['fenton_phe_removal_pct']:>10.1f}")
print()
print(f"Carbon saving from bioremediation: {comparison['carbon_saving_tCO2']:.3f} tCO2eq")

Site volume: 2,000 m3

                                     Bioremediation     Fenton
--------------------------------------------------------------
Carbon footprint (tCO2eq)                     0.264      1.211
Hexadecane removal (%)                        97.24       90.0
Phenanthrene removal (%)                      63.44       80.0

Carbon saving from bioremediation: 0.947 tCO2eq
